In [1]:
import scvelo as scv
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import os, sys

os.chdir("/mnt/yijun/nfs_share/awa_project/awa_github/TemporalVAE/")
scv.settings.set_figure_params('scvelo')

import matplotlib.colors as mcolors

color_plate = list(mcolors.TABLEAU_COLORS)


In [2]:
### option 1: calculate RNA velocity from the beginning.
adata = sc.read("data/240108mouse_embryogenesis/blood_exp.mtx", cache=True)
spliced = sc.read("data/240108mouse_embryogenesis/blood_exp_exon.mtx", cache=True)
unspliced = sc.read("data/240108mouse_embryogenesis/blood_exp_intron.mtx", cache=True)
pdata = pd.read_csv("data/240108mouse_embryogenesis/blood_obs.csv", index_col=0)
fdata = pd.read_csv("data/240108mouse_embryogenesis/blood_var.csv", index_col=0)

### add spliced and unspliced to layers, pd to obs
adata.layers['spliced'] = spliced.X
adata.layers['unspliced'] = unspliced.X
adata.obs = pdata
adata.var = fdata
scv.utils.show_proportions(adata)

Abundance of ['unspliced', 'spliced']: [0.62 0.38]


In [3]:
print("Import data, cell number: {}, gene number: {}".format(adata.n_obs, adata.n_vars))
print("Annotation information of data includes: {}".format(adata.obs_keys()))  # 胞注釋信息的keys
print("Cell id first 5: {}".format(adata.obs_names[:5]))  # 返回胞ID 数据类型是object
print("Gene id first 5: {}".format(adata.var_names[:5]))  # 返回基因数据类型是list adata.obs.head()# 査看前5行的数据
print(adata.obs.celltype.value_counts())
print(adata.obs.day.value_counts())
print(adata.X)
print(adata.obs)

Import data, cell number: 53268, gene number: 24552
Annotation information of data includes: ['Anno', 'day', 'celltype', 'sample', 'batch', 'group']
Cell id first 5: Index(['P2-01A.ATGGTAACTTAGCCGGTACC', 'P2-01A.AACGAGCGTCCGTTCGGAT',
       'P2-01A.CCGTCGATTCTTGCAACCT', 'P2-01A.ATTGAGGAATTTATTCTGAG',
       'P2-01A.ATTCGGAGTTATAGACGCA'],
      dtype='object')
Gene id first 5: Index(['ENSMUSG00000051951', 'ENSMUSG00000102343', 'ENSMUSG00000025900',
       'ENSMUSG00000025902', 'ENSMUSG00000104328'],
      dtype='object')
celltype
Definitive erythroid cells    22038
Primitive erythroid cells     21309
White blood cells              8213
Megakaryocytes                 1509
Blood progenitors               199
Name: count, dtype: int64
day
E11.5    14930
E13.5    13673
E10.5     9308
E12.5     9090
E9.5      3390
E8.5b     2877
Name: count, dtype: int64
  (0, 8)	1.0
  (0, 25)	1.0
  (0, 26)	2.0
  (0, 28)	1.0
  (0, 31)	1.0
  (0, 32)	1.0
  (0, 33)	2.0
  (0, 55)	1.0
  (0, 56)	1.0
  (0, 122)	1.0

In [4]:
gene_list_atlas = pd.read_csv("data/mouse_embryonic_development//preprocess_adata_JAX_dataset_combine_minGene100_minCell50_hvg1000/gene_info.csv",
                              index_col=0, sep='\t')
gene_list_atlas['gene_id']

ENSMUSG00000002459    ENSMUSG00000002459
ENSMUSG00000033740    ENSMUSG00000033740
ENSMUSG00000025909    ENSMUSG00000025909
ENSMUSG00000025915    ENSMUSG00000025915
ENSMUSG00000046101    ENSMUSG00000046101
                             ...        
ENSMUSG00000063434    ENSMUSG00000063434
ENSMUSG00000043531    ENSMUSG00000043531
ENSMUSG00000035804    ENSMUSG00000035804
ENSMUSG00000043639    ENSMUSG00000043639
ENSMUSG00000025089    ENSMUSG00000025089
Name: gene_id, Length: 979, dtype: object

In [5]:
adata = adata[:, gene_list_atlas['gene_id']]
adata

View of AnnData object with n_obs × n_vars = 53268 × 979
    obs: 'Anno', 'day', 'celltype', 'sample', 'batch', 'group'
    var: 'gene_id', 'gene_short_name'
    layers: 'spliced', 'unspliced'

In [6]:
import scanpy as sc

# sc.pp.filter_cells(adata,min_genes=20)
scv.pp.filter_genes(adata, min_counts=5, min_counts_u=5)
scv.pp.normalize_per_cell(adata)
adata.raw = adata
# scv.pp.filter_genes_dispersion(adata, n_top_genes=15000) # orginal 3000
sc.pp.log1p(adata)

Filtered out 147 genes that are detected 5 counts (spliced).
Filtered out 65 genes that are detected 5 counts (unspliced).
Normalized count data: X, spliced, unspliced.


In [7]:
print("Import data, cell number: {}, gene number: {}".format(adata.n_obs, adata.n_vars))
print("Annotation information of data includes: {}".format(adata.obs_keys()))  # 胞注釋信息的keys
print("Cell id first 5: {}".format(adata.obs_names[:5]))  # 返回胞ID 数据类型是object
print("Gene id first 5: {}".format(adata.var_names[:5]))  # 返回基因数据类型是list adata.obs.head()# 査看前5行的数据
print(adata.obs.celltype.value_counts())
print(adata.obs.day.value_counts())
print(adata.X)
print(adata.obs)

Import data, cell number: 53268, gene number: 767
Annotation information of data includes: ['Anno', 'day', 'celltype', 'sample', 'batch', 'group', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'n_counts']
Cell id first 5: Index(['P2-01A.ATGGTAACTTAGCCGGTACC', 'P2-01A.AACGAGCGTCCGTTCGGAT',
       'P2-01A.CCGTCGATTCTTGCAACCT', 'P2-01A.ATTGAGGAATTTATTCTGAG',
       'P2-01A.ATTCGGAGTTATAGACGCA'],
      dtype='object')
Gene id first 5: Index(['ENSMUSG00000002459', 'ENSMUSG00000033740', 'ENSMUSG00000025909',
       'ENSMUSG00000025915', 'ENSMUSG00000046101'],
      dtype='object')
celltype
Definitive erythroid cells    22038
Primitive erythroid cells     21309
White blood cells              8213
Megakaryocytes                 1509
Blood progenitors               199
Name: count, dtype: int64
day
E11.5    14930
E13.5    13673
E10.5     9308
E12.5     9090
E9.5      3390
E8.5b     2877
Name: count, dtype: int64
  (0, 3)	0.6074917316436768
  (0, 29)	0.6074917316436768
  (0, 

In [8]:
def geneId_geneName_dic(return_total_gene_pd_bool=False):
    gene_data = pd.read_csv("data/mouse_embryonic_development//df_gene.csv", index_col=0)
    gene_dict = gene_data.set_index('gene_id')['gene_short_name'].to_dict()
    if return_total_gene_pd_bool:
        return gene_dict, gene_data
    else:
        return gene_dict


gene_dic = geneId_geneName_dic()
adata.var_names = [gene_dic.get(name, name) for name in adata.var_names]


In [9]:
print("Import data, cell number: {}, gene number: {}".format(adata.n_obs, adata.n_vars))
print("Annotation information of data includes: {}".format(adata.obs_keys()))  # 胞注釋信息的keys
print("Cell id first 5: {}".format(adata.obs_names[:5]))  # 返回胞ID 数据类型是object
print("Gene id first 5: {}".format(adata.var_names[:5]))  # 返回基因数据类型是list adata.obs.head()# 査看前5行的数据

Import data, cell number: 53268, gene number: 767
Annotation information of data includes: ['Anno', 'day', 'celltype', 'sample', 'batch', 'group', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'n_counts']
Cell id first 5: Index(['P2-01A.ATGGTAACTTAGCCGGTACC', 'P2-01A.AACGAGCGTCCGTTCGGAT',
       'P2-01A.CCGTCGATTCTTGCAACCT', 'P2-01A.ATTGAGGAATTTATTCTGAG',
       'P2-01A.ATTCGGAGTTATAGACGCA'],
      dtype='object')
Gene id first 5: Index(['Rgs20', 'St18', 'Sntg1', 'Sgk3', 'Mcmdc2'], dtype='object')


In [10]:
adata.var_names_make_unique()
adata

AnnData object with n_obs × n_vars = 53268 × 767
    obs: 'Anno', 'day', 'celltype', 'sample', 'batch', 'group', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'n_counts'
    var: 'gene_id', 'gene_short_name'
    uns: 'log1p'
    layers: 'spliced', 'unspliced'

In [11]:
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
adata

/mnt/yijun/nfs_share/yijun_tmp/ipykernel_1989048/400086297.py:1: DeprecationWarning: Automatic neighbor calculation is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors first with Scanpy.
  scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
/mnt/yijun/nfs_share/miniconda3/envs/temporalVAE_github/lib/python3.10/site-packages/scvelo/preprocessing/moments.py:71: DeprecationWarning: `neighbors` is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors with Scanpy.
  neighbors(
/mnt/yijun/nfs_share/miniconda3/envs/temporalVAE_github/lib/python3.10/site-packages/scvelo/preprocessing/neighbors.py:233: DeprecationWarning: Automatic computation of PCA is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute PCA with Scanpy first.
  _set_pca(adata=adata, n_pcs=n_pcs, use_highly_variable=use_highly_variable)


computing neighbors


2024-10-09 15:00:39.975572: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


    finished (0:00:51) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:01) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)


AnnData object with n_obs × n_vars = 53268 × 767
    obs: 'Anno', 'day', 'celltype', 'sample', 'batch', 'group', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'n_counts'
    var: 'gene_id', 'gene_short_name'
    uns: 'log1p', 'pca', 'neighbors'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: 'spliced', 'unspliced', 'Ms', 'Mu'
    obsp: 'distances', 'connectivities'

In [12]:
adata.write_h5ad("data/240108mouse_embryogenesis/hematopoiesis.h5ad")
print(f"Preprocessed data save as data/240108mouse_embryogenesis/hematopoiesis.h5ad")

Preprocessed data save as data/240108mouse_embryogenesis/hematopoiesis.h5ad
